In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
import torch.nn.functional as F 

import os
from tqdm import tqdm
from data_loader import get_train_test_loaders, get_unlabeled_loader


In [ ]:
# Hyperparameters
epochs = 5
learning_rate = 0.001
batch_size = 16
num_classes = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Load train/test data
train_loader, test_loader, train_set, test_set = get_train_test_loaders(batch_size=batch_size)

# Load and modify pretrained Efficient Net B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=True, unit='batch')
    for i, (images, labels, path) in enumerate(progress_bar):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        avg_loss = running_loss / (i + 1)
        progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
# --- Save the Model ---
# Create the directory if it doesn't exist
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')
    print(f"Created directory: {'saved_models'}")

# Construct the full path for saving the model
full_save_path = os.path.join('saved_models', 'efficientnet_b0_model.pth')

# Save the model's state dictionary
torch.save(model.state_dict(), full_save_path)
print(f'Model state dictionary saved to: {full_save_path}')

In [ ]:
# Evaluation
model.load_state_dict(torch.load('saved_models/efficientnet_b0_model_5epochs.pth'))
model.eval()
correct = 0
total = 0
incorrect_preds = []

with torch.no_grad():
    for images, labels, paths in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 

        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for i in range(len(labels)):
            if predicted[i] != labels[i]:
                incorrect_preds.append({
                    'path': paths[i],
                    'true_label': labels[i].item(),
                    'predicted_label': predicted[i].item(),
                    'confidence': probs[i][predicted[i]].item()
                })

# Combine
all_probs_tensor = torch.cat(all_probs, dim=0)

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

print('Files with incorrect predictions:')
for item in incorrect_preds:
    print(f"{item['path']} -> True: {item['true_label']}, Pred: {item['predicted_label']}, Confidence: {item['confidence']:.4f}")

In [ ]:
# Unlabeled Data
unlabeled_set, _ = get_unlabeled_loader(batch_size=batch_size)

# Evaluation
model.eval()
all_probs = []
all_paths = []
with torch.no_grad():
    for images, paths in unlabeled_set:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 
        all_probs.append(probs.cpu())
        all_paths.extend(paths)


# Combine
all_probs_tensor = torch.cat(all_probs, dim=0)

# Get max probs and predicted classes
max_probs, predicted_classes = torch.max(all_probs_tensor, dim=1)

# Confidence filtering
threshold = 0.95
confident_mask = max_probs >= threshold

confident_probs = max_probs[confident_mask]
confident_preds = predicted_classes[confident_mask]
confident_paths = [all_paths[i] for i in range(len(all_paths)) if confident_mask[i]]

# Report
print(f'Found {len(confident_preds)} confident predictions:')
for path, pred, prob in zip(confident_paths, confident_preds, confident_probs):
    print(f'{path} -> Class {pred.item()} with probability {prob.item():.4f}')